# 02 – Data Cleaning

## 1. Project Objective

The objective of this notebook is to prepare reliable, consistent, and analysis-ready datasets for the Decathlon Retail Performance Analytics project.

The cleaning process will preserve valid business transactions while addressing only genuine data-quality issues identified during the Data Inspection and Validation phase.

## 2. Source Datasets

The following raw datasets will be used:

| Dataset | Description |
|---|---|
| `sales.xlsx` | Sales transaction-line data |
| `products.xlsx` | Product master data |
| `store.xlsx` | Store master data |

**Source folder:** `Data/Raw/`

## 3. Scope of Data Cleaning

The cleaning process will cover:

1. Creating working copies of the raw datasets.
2. Standardizing column names and text values where required.
3. Ensuring appropriate data types.
4. Handling missing values based on business meaning.
5. Checking duplicate records and key consistency.
6. Preserving valid returns and fully refunded transactions.
7. Validating financial calculations and business rules.
8. Preparing clean datasets for PostgreSQL analysis.

## 4. Important Business Rules

- Raw source files must remain unchanged.
- Missing `Promotion` values must not automatically be replaced with `No Promotion`.
- Fully refunded transactions with zero net sales must be preserved.
- Existing discounts and refunds must not be deducted again.
- Transaction prices and costs must not be overwritten using product catalogue values.
- Net units sold = `Quantity - Returned_Qty`.
- Total orders must be calculated using distinct `Order_ID`.

In [2]:
# Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path

# Locate project root

current_path = Path.cwd()

project_root = next(
    (path for path in [current_path, *current_path.parents]
     if (path / "Data" / "Raw").exists()),
    None
)

if project_root is None:
    raise FileNotFoundError("Project folder not found.")

# Define source folder

raw_data_path = project_root / "Data" / "Raw"

# Load raw datasets into working DataFrames

sales_df = pd.read_excel(raw_data_path / "sales.xlsx")
products_df = pd.read_excel(raw_data_path / "products.xlsx")
store_df = pd.read_excel(raw_data_path / "store.xlsx")

print("All three datasets loaded successfully.")

All three datasets loaded successfully.


In [3]:
# Create working copies for data cleaning

sales_clean = sales_df.copy()
products_clean = products_df.copy()
store_clean = store_df.copy()

print("Working copies created successfully.")

print("\nDataset Dimensions:")
print("Sales:", sales_clean.shape)
print("Products:", products_clean.shape)
print("Store:", store_clean.shape)

Working copies created successfully.

Dataset Dimensions:
Sales: (84000, 22)
Products: (240, 9)
Store: (30, 8)


In [4]:
# Standardize column names across all datasets

sales_clean.columns = (
    sales_clean.columns.str.strip().str.lower()
    .str.replace(" ", "_")
)

products_clean.columns = (
    products_clean.columns.str.strip().str.lower()
    .str.replace(" ", "_")
)

store_clean.columns = (
    store_clean.columns.str.strip().str.lower()
    .str.replace(" ", "_")
)

print("Column names standardized successfully.")

print("\nSales Columns:")
print(sales_clean.columns.tolist())

print("\nProducts Columns:")
print(products_clean.columns.tolist())

print("\nStore Columns:")
print(store_clean.columns.tolist())

Column names standardized successfully.

Sales Columns:
['sale_line_id', 'order_id', 'purchase_date', 'product_id', 'store_id', 'channel', 'payment_method', 'order_hour', 'promotion', 'quantity', 'unit_list_price_inr', 'unit_cost_inr', 'discount_pct', 'gross_sales_inr', 'discount_inr', 'sales_before_returns_inr', 'returned_qty', 'refund_inr', 'net_sales_inr', 'net_cogs_inr', 'gross_profit_inr', 'return_reason']

Products Columns:
['product_key', 'product_name', 'sport', 'product_type', 'brand_label', 'product_tier', 'colour', 'base_list_price_inr', 'base_unit_cost_inr']

Store Columns:
['store_key', 'store_name', 'city', 'state', 'region', 'country', 'store_format', 'floor_area_sqm']


In [5]:
# Identify text columns in each dataset

sales_text_columns = sales_clean.select_dtypes(include="object").columns
products_text_columns = products_clean.select_dtypes(include="object").columns
store_text_columns = store_clean.select_dtypes(include="object").columns

# Clean text values without changing missing values

for column in sales_text_columns:
    sales_clean[column] = sales_clean[column].str.strip()

for column in products_text_columns:
    products_clean[column] = products_clean[column].str.strip()

for column in store_text_columns:
    store_clean[column] = store_clean[column].str.strip()

print("Text values standardized successfully.")

C:\Users\Dell\AppData\Local\Temp\ipykernel_36244\2953591226.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  sales_text_columns = sales_clean.select_dtypes(include="object").columns
C:\Users\Dell\AppData\Local\Temp\ipykernel_36244\2953591226.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/d

Text values standardized successfully.


In [6]:
print("Sales Data Types:")
print(sales_clean.dtypes)

print("\nProducts Data Types:")
print(products_clean.dtypes)

print("\nStore Data Types:")
print(store_clean.dtypes)

Sales Data Types:
sale_line_id                           str
order_id                               str
purchase_date               datetime64[us]
product_id                             str
store_id                               str
channel                                str
payment_method                         str
order_hour                           int64
promotion                              str
quantity                             int64
unit_list_price_inr                float64
unit_cost_inr                      float64
discount_pct                       float64
gross_sales_inr                    float64
discount_inr                       float64
sales_before_returns_inr           float64
returned_qty                         int64
refund_inr                         float64
net_sales_inr                      float64
net_cogs_inr                       float64
gross_profit_inr                   float64
return_reason                          str
dtype: object

Products Data Types:


In [7]:
processed_path = project_root / "Data" / "Processed"
processed_path.mkdir(parents=True, exist_ok=True)

sales_clean.to_excel(processed_path / "sales_clean.xlsx", index=False)
products_clean.to_excel(processed_path / "products_clean.xlsx", index=False)
store_clean.to_excel(processed_path / "store_clean.xlsx", index=False)

print("All three cleaned datasets saved successfully.")

All three cleaned datasets saved successfully.


In [8]:
# Convert cleaned Excel datasets to CSV

sales_clean.to_csv(processed_path / "sales_clean.csv", index=False)
products_clean.to_csv(processed_path / "products_clean.csv", index=False)
store_clean.to_csv(processed_path / "store_clean.csv", index=False)

print("All three CSV files created successfully.")

All three CSV files created successfully.


# Data Cleaning – Overall Summary

## 1. Project Overview

The objective of this phase was to prepare the Decathlon retail datasets for SQL business analysis and Power BI reporting.

Three datasets were processed:

- Sales
- Products
- Store

## 2. Cleaning Activities Completed

| Activity | Result |
|---|---|
| Column name standardization | Completed |
| Text whitespace removal | Completed |
| Missing-value review | Completed |
| Duplicate review | Completed |
| Data type review | Completed |
| Primary and foreign key review | Previously validated |
| Business-rule review | Previously validated |
| Financial reconciliation | Previously validated |

## 3. Important Data Handling Decisions

- Missing Promotion values were preserved.
- Fully refunded transactions were retained.
- No valid transaction records were intentionally removed.
- Existing transaction prices and costs were preserved.
- Existing financial calculations were retained.
- Original raw Excel files were not modified.

## 4. Processed Dataset Summary

| Dataset | Rows | Columns |
|---|---:|---:|
| Sales | 84,000 | 22 |
| Products | 240 | 9 |
| Store | 30 | 8 |

## 5. Output Files

The cleaned datasets were saved in `Data/Processed/`.

- `sales_clean.xlsx`
- `products_clean.xlsx`
- `store_clean.xlsx`